<a href="https://colab.research.google.com/github/LFZ121/UDL-notebook/blob/main/Notebooks/Chap13/13_4_Graph_Attention_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Notebook 13.4: Graph attention networks**

This notebook builds a graph attention mechanism from scratch, as discussed in section 13.8.6 of the book and illustrated in figure 13.12c

Work through the cells below, running each cell in turn. In various places you will see the words "TODO". Follow the instructions at these places and make predictions about what is going to happen or write code to complete the functions.

Contact me at udlbookmail@gmail.com if you find any mistakes or have any suggestions.



In [1]:
import numpy as np
import matplotlib.pyplot as plt

The self-attention mechanism maps $N$ inputs $\mathbf{x}_{n}\in\mathbb{R}^{D}$ and returns $N$ outputs $\mathbf{x}'_{n}\in \mathbb{R}^{D}$.  



In [2]:
# Set seed so we get the same random numbers
np.random.seed(1)
# Number of nodes in the graph
N = 8
# Number of dimensions of each input
D = 4

# Define a graph
A = np.array([[0,1,0,1,0,0,0,0],
              [1,0,1,1,1,0,0,0],
              [0,1,0,0,1,0,0,0],
              [1,1,0,0,1,0,0,0],
              [0,1,1,1,0,1,0,1],
              [0,0,0,0,1,0,1,1],
              [0,0,0,0,0,1,0,0],
              [0,0,0,0,1,1,0,0]]);
print(A)

# Let's also define some random data
X = np.random.normal(size=(D,N))

[[0 1 0 1 0 0 0 0]
 [1 0 1 1 1 0 0 0]
 [0 1 0 0 1 0 0 0]
 [1 1 0 0 1 0 0 0]
 [0 1 1 1 0 1 0 1]
 [0 0 0 0 1 0 1 1]
 [0 0 0 0 0 1 0 0]
 [0 0 0 0 1 1 0 0]]


We'll also need the weights and biases for the keys, queries, and values (equations 12.2 and 12.4)

In [3]:
# Choose random values for the parameters
omega = np.random.normal(size=(D,D))
beta = np.random.normal(size=(D,1))
phi = np.random.normal(size=(2*D,1))

We'll need a softmax operation that operates on the columns of the matrix and a ReLU function as well

In [4]:
# Define softmax operation that works independently on each column
def softmax_cols(data_in):
  # Exponentiate all of the values
  exp_values = np.exp(data_in) ;
  # Sum over columns
  denom = np.sum(exp_values, axis = 0);
  # Replicate denominator to N rows
  denom = np.matmul(np.ones((data_in.shape[0],1)), denom[np.newaxis,:])
  # Compute softmax
  softmax = exp_values / denom
  # return the answer
  return softmax


# Define the Rectified Linear Unit (ReLU) function
def ReLU(preactivation):
  activation = preactivation.clip(0.0)
  return activation


In [5]:
 # Now let's compute self attention in matrix form
def graph_attention(X, omega, beta, phi, A):
    D, N = X.shape

    # 1. 计算 X_prime: 对每个节点应用线性变换 (Omega * x + beta)
    # 使用 np.ones 来广播偏置项 beta
    X_prime = omega @ X + beta @ np.ones((1, N))

    # 2. 计算得分矩阵 S (N x N)
    S = np.zeros((N, N))
    for n in range(N): # 目标节点 (Query)
        for m in range(N): # 来源节点 (Key)
            # 按照图 13.12c，将 X'_m 和 X'_n 拼接成 2D 维向量
            val_m = X_prime[:, m:m+1]
            val_n = X_prime[:, n:n+1]
            concat_vec = np.vstack((val_m, val_n))

            # 使用向量 phi 计算注意力得分
            S[m, n] = phi.T @ concat_vec

    # 3. 应用掩码 (Masking):
    # 只允许在图中有连接的邻居（包括自身 A+I）之间进行注意
    mask = A + np.eye(N)
    S[mask == 0] = -1e20 # 设置为极小的负数，使 softmax 后的权重接近 0

    # 4. 运行 softmax 函数计算注意力权重
    attn = softmax_cols(S)

    # 5. 将 X' 与注意力权重相乘 (加权求和)
    # 每个新特征是邻居特征的加权组合
    X_output = X_prime @ attn

    # 6. 应用 ReLU 激活函数
    output = ReLU(X_output)

    return output

In [6]:
# Test out the graph attention mechanism
np.set_printoptions(precision=3)
output = graph_attention(X, omega, beta, phi, A);
print("Correct answer is:")
print("[[0.    0.028 0.37  0.    0.97  0.    0.    0.698]")
print(" [0.    0.    0.    0.    1.184 0.    2.654 0.  ]")
print(" [1.13  0.564 0.    1.298 0.268 0.    0.    0.779]")
print(" [0.825 0.    0.    1.175 0.    0.    0.    0.  ]]]")


print("Your answer is:")
print(output)

Correct answer is:
[[0.    0.028 0.37  0.    0.97  0.    0.    0.698]
 [0.    0.    0.    0.    1.184 0.    2.654 0.  ]
 [1.13  0.564 0.    1.298 0.268 0.    0.    0.779]
 [0.825 0.    0.    1.175 0.    0.    0.    0.  ]]]
Your answer is:
[[0.    0.    0.    0.    0.09  0.    0.    0.673]
 [0.    0.    0.    0.    0.    0.    0.    0.   ]
 [1.343 1.122 0.    1.303 0.    0.    0.    0.763]
 [1.344 0.983 0.    1.186 0.    0.    0.    0.   ]]


/tmp/ipykernel_25018/183897574.py:19: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  S[m, n] = phi.T @ concat_vec


TODO -- Try to construct a dot-product self-attention mechanism as in practical 12.1 that respects the geometry of the graph and has zero attention between non-neighboring nodes by combining figures 13.12a and 13.12b.


1,In a standard GNN, neighbors are usually aggregated using a fixed operation (like a simple mean or sum), treating all neighbors as equally important. In a GAT, we learn a weight (attention) for each neighbor, allowing the model to focus more on relevant parts of the neighborhood for a specific task.

2,This is the masking step. It ensures that the model only pays attention to immediate neighbors defined by the graph structure. By setting non-neighbor scores to $-10^{20}$, the $exp(S)$ term in the softmax becomes zero, effectively forcing the attention weight for those nodes to be zero.
